In [1]:
API_KEY = ""

## 2.1 Anthropic SDK 설치

In [ ]:
!pip install anthropic -qq

## 2.2 Anthropic messages API 파라미터 설정 예시

In [2]:
import anthropic

client = anthropic.Anthropic(
    api_key=API_KEY,
)

message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=50,
    temperature=0.02,
    system="주어진 문장을 이어서 10개 단어 이내의 완벽한 문장으로 작성해줘.",
    messages=[{"role": "user", "content": f"문장: 나는 오늘 학교에 가서"}]
)

print(message.content)

[TextBlock(text='나는 오늘 학교에 가서 친구들과 즐겁게 점심을 먹었다.', type='text')]


## 2.3 문장 감정 분류 예시

```
"음식은 괜찮았던 것 같아요" 문장을 중립/부정/긍정 으로 분류해줘.
```

## 2.4 명확한 작업 설명과 출력 형식을 추가한 예시

```
주어진 문장을 긍정, 부정 또는 중립으로만 분류합니다. 분류에 대한 추가 설명을 하지 않습니다.  
문장: 음식은 괜찮았던 것 같아요.  
분류:
```

## 2.5 세부 지침이 포함된 프롬프트

```
주어진 문장을 지침을 따라 A, B, C 로 분류합니다.  
분류에 대한 추가 설명을 하지 않습니다.   

다음 지침을 따라 문장을 분류합니다.  
- A: 문장이 긍정적인 감정을 나타낼 때  
- B: 문장이 부정적인 감정을 나타낼 때  
- C: 문장에서 긍정이나 부정의 강한 감정이 없을 때  

문장: 음식은 좋아.  
분류:
```

## 2.6 시스템 메시지

```json
{"role": "system", "content": "주어진 문장을 긍정, 부정 또는 중립으로만 분류합니다. 분류에 대한 추가 설명을 하지 않습니다."}
```

## 2.7 메시지

```json
[
	{"role": "system", "content": "주어진 문장을 긍정, 부정 또는 중립으로만 분류합니다. 분류에 대한 추가 설명을 하지 않습니다."}
	{"role": "user", "content": "문장: 음식은 괜찮았던 것 같아요.\n분류:"
]
```

## 2.8 anthropic 을 통해 assistant 발화 생성

In [ ]:
import anthropic

client = anthropic.Anthropic(
    api_key=API_KEY
)

message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=10,
    temperature=0.0,
    system="주어진 문장을 긍정, 부정 또는 중립으로만 분류합니다. 분류에 대한 추가 설명을 하지 않습니다.",
    messages=[
        {"role": "user", "content": "문장: 음식은 괜찮았던 것 같아요.\n분류:"}
    ]
)

print(f"{message.role}: {message.content[0].text}")
# assistant: 중립

## 2.9 맥락을 이용한 답변 생성 예시

In [ ]:
message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=100,
    temperature=0.0,
    system="당신은 사용자의 친구로 대화를 이어나갑니다. 주어진 맥락을 이용하여 답변합니다.",
    messages=[
        {"role": "user", "content": "안녕? 내 이름은 슬기야! 네 이름은 뭐야?"},
        {"role": "assistant", "content": "안녕? 내 이름은 유정이야."},
        {"role": "user", "content": "응? 잘 못 들었어. 이름이 뭐라고?"}
    ]
)

print(f"{message.role}: {message.content[0].text}")
# assistant: 아, 내 이름은 유정이라고 했어! 만나서 반가워 슬기야~

## 2.10 문장 감정 분류 프롬프트 템플릿

```
주어진 문장을 긍정, 부정 또는 중립으로만 분류합니다.

문장: [문장 내용]  
분류:
```

## 2.11 프롬프트 템플릿을 이용한 반복 작업 자동화 예시

In [ ]:
def create_prompt(sentence):
    return {
        "system": "주어진 문장을 긍정, 부정, 중립으로 분류합니다. 분류에 대한 추가 설명은 제공하지 않습니다.",
        "messages": [
            {"role": "user", "content": f"문장: {sentence}\n분류:"}
        ]
    }

sentences = [
    "음식은 괜찮았던 것 같아요.",
    "서비스가 별로였어요.",
    "매장이 깔끔하고 좋았어요."
]

prompts = [create_prompt(sentence) for sentence in sentences]

## 2.12 프롬프트 템플릿을 통한 효율적인 응답 생성 예시

In [ ]:
outputs = {
    sentence: client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=100,
        temperature=0.0,
        **create_prompt(sentence)
    ).content[0].text
    for sentence in sentences
}

for sentence, category in outputs.items():
    print(f"문장: {sentence} 분류: {category}")
    
# 문장: 음식은 괜찮았던 것 같아요. 분류: 중립
# 문장: 서비스가 별로였어요. 분류: 부정
# 문장: 매장이 깔끔하고 좋았어요. 분류: 긍정